# Avance 6 - Demostracion del Copiloto Conversacional

### Patron "Be My Eyes": los modelos del equipo *ven*, un LLM frontera *razona*

**Equipo 17** - AgroSatCopilot

---

Este cuaderno demuestra, de principio a fin y **con datos reales**, el copiloto conversacional para analisis satelital agricola. La idea central es el patron **Be My Eyes**:

- El **perceiver** son los modelos entrenados por el equipo (el ensamble campeon **Voting-3**: tsvit-pheno + utae + xgb-alphaearth, y el descriptor fenologico). No hablan con el usuario: **miran una parcela y emiten una observacion en TEXTO** (cultivo, fenologia, vigor, confianza).
- El **reasoner** es un LLM frontera (Gemini en la nube) u on-prem (Qwen). **No clasifica pixeles**: lee ese texto, llama herramientas geoespaciales cuando hace falta y redacta la respuesta en lenguaje natural.

Esta separacion es lo que hace al sistema **auditable y anti-alucinacion**: toda cifra que el reasoner enuncia proviene de una herramienta o de una observacion del perceiver, nunca de la imaginacion del modelo.

> Esta es la demo **general** del copiloto sobre las parcelas PASTIS sembradas. El transfer learning mediterraneo (US-079, original vs Italia) se trata aparte, en `notebooks/transfer/` (`us079_copilot_original_vs_tl` y `us079_transfer_italia_eval`), para que este cuaderno se mantenga centrado en el mecanismo.

> Corre contra la sesion de demostracion ya sembrada en la base local (Postgres + PostGIS + pgvector): parcelas reales de **PASTIS-R** (fold-5 retenido) que el campeon **Voting-3** puntua desde su OOF real, y un corpus de descripciones fenologicas (FarSLIP) con vector para el RAG.

In [1]:
# Parameters cell (papermill). Defaults are the seeded demo values; override
# any of them at run time with `papermill -p <name> <value>`.
model = 'gemini-3.5-flash'        # default cloud reasoner (live if GEMINI_API_KEY set)
demo_user = 'demo@agrosat.dev'    # seeded demo session owner
n_perceiver_parcels = 3           # how many parcels to run through the perceiver
classify_year = 2019              # AlphaEarth annual campaign for classify / aoi_stats
classify_model = 'voting3'        # EPIC 12 deployment champion served by classify
rag_radius_m = 20000.0            # ST_DWithin radius for the Spatial-RAG demo (m)
rag_top_k = 5                     # documents retrieved per RAG query
# The three reasoner backends contrasted in section 5 (make_backend resolves each).
backend_models = ['gemini-3.5-flash', 'qwen3.6-vl', 'qwen35']

## Preparacion del entorno

Resolvemos la raiz del repositorio (sin rutas absolutas), cargamos `.env.local` para tomar la cadena de conexion y las credenciales del LLM, y **silenciamos el ruido de logs** del agente (structlog emite una linea INFO por paso; en una demo en vivo eso tapa la respuesta). La consola de Windows usa cp1252; forzamos UTF-8 en la salida para que los acentos no rompan la ejecucion.

In [2]:
# --- Repo bootstrap, UTF-8 safety, env, autoreload, quiet logging ---
import os
import sys
from pathlib import Path

# Windows console is cp1252; structlog and Spanish prose use accents. Reconfigure
# stdout/stderr to UTF-8 so an accented log line never raises UnicodeEncodeError.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding='utf-8')
    except (AttributeError, ValueError):
        pass

from ml.utils.notebook_setup import find_repo_root, load_env_local

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
load_env_local(REPO_ROOT)

# Hot-reload so edits in ml/*.py are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import time

import polars as pl
from IPython.display import Markdown, display

# Reusable copilot-demo driver + renderers (unit-tested in tests/ml/agent/test_demo).
# Keeping them in ml/agent/demo is what lets every cell below be a single short call.
from ml.agent import demo

# Silence the agent/perceiver/tool INFO-DEBUG structlog stream (warnings still show).
demo.quiet_logging()

print('repo:', REPO_ROOT)
print('reasoner por defecto:', model, '| usuario demo:', demo_user)

repo: C:\Users\arthu\Proyectos\MNA\agro_sat_copilot
reasoner por defecto: gemini-3.5-flash | usuario demo: demo@agrosat.dev


In [3]:
# --- Connect to the seeded demo session ---
from backend.app.core.config import get_settings
from ml.agent.db import get_pool

settings = get_settings()
pool = await get_pool()

async with pool.acquire() as conn:
    # Pick the demo user's session that actually owns parcels (deterministic even
    # if several demo sessions exist), so the perceiver/tools have data to work on.
    session_id = await conn.fetchval(
        'SELECT cs.id FROM chat_sessions cs '
        'LEFT JOIN parcels p ON p.session_id = cs.id '
        'WHERE cs.user_id = $1 '
        'GROUP BY cs.id ORDER BY count(p.id) DESC, cs.id LIMIT 1',
        demo_user,
    )
    n_parcels = await conn.fetchval('SELECT count(*) FROM parcels')
    n_features = await conn.fetchval('SELECT count(*) FROM features_parcels')
    n_rag = await conn.fetchval('SELECT count(*) FROM rag_documents')

assert session_id is not None, (
    f'no demo session for user {demo_user!r}; seed the demo data first'
)
display(Markdown(
    f'**Sesion de demostracion**: `{session_id}`  \n'
    f'Parcelas sembradas: **{n_parcels}** | features de parcela: **{n_features}** | '
    f'documentos RAG: **{n_rag}**'
))

**Sesion de demostracion**: `f1417dbc-3053-413e-8c5a-b66f9b748c0d`  
Parcelas sembradas: **10** | features de parcela: **0** | documentos RAG: **300**

In [4]:
# --- Shared tool-execution context (multi-tenant: scoped to this session) ---
from ml.agent.context import ToolContext

ctx = ToolContext(pool=pool, settings=settings, session_id=session_id)
print('ToolContext listo | session_id =', ctx.session_id)

ToolContext listo | session_id = f1417dbc-3053-413e-8c5a-b66f9b748c0d


## 1. Las herramientas geoespaciales

El reasoner no accede a la base de datos por su cuenta: actua **solo** a traves de un conjunto cerrado de herramientas geoespaciales, cada una con un esquema de entrada y salida validado (Pydantic). Esto acota lo que el agente puede hacer y deja cada accion rastreable.

Cinco son **sincronas** (se ejecutan en linea dentro del bucle del agente: listar parcelas, serie temporal, estadisticas de un area, clasificar y explicar). Las otras cinco son **diferidas** (*deferred*): se completan fuera de linea via un *worker* (buscar escenas, teselas de mapa, guardar un area, comparar modelos y recuperar contexto del RAG). La tabla se construye desde `build_function_declarations()` -- la misma fuente de verdad que se le anuncia al LLM.

In [5]:
# The tool table, straight from the declarations advertised to the LLM. The whole
# rendering lives in ml/agent/demo so this cell is one line.
_tools = demo.tool_inventory()

herramienta,tipo,comportamiento,descripcion
str,str,str,str
"""add_aoi""","""diferida""","""NON_BLOCKING""","""Persist a named Area Of Interest polygon for the current session."""
"""compare_models""","""diferida""","""NON_BLOCKING""","""Compare the crop predictions of several ensemble members for one parcel."""
"""get_tiles""","""diferida""","""NON_BLOCKING""","""Build a TiTiler XYZ tile-template URL for a scene rendered as an index or RGB."""
"""retrieve_context""","""diferida""","""NON_BLOCKING""","""Retrieve real neighbouring-parcel grounding (Spatial-RAG lite) for an AOI; gated by the rag_enabled flag (no-op when off…"
"""search_stac""","""diferida""","""NON_BLOCKING""","""Search Sentinel-2 scenes in a STAC catalogue by bbox, datetime range and cloud cover."""
"""classify_new_parcel""","""sincrona""","""BLOCKING""","""Classify a parcel's crop. By default (model='xgb') serves the xgb-alphaearth tabular member restricted to the active lab…"
"""explain_prediction""","""sincrona""","""BLOCKING""","""Explain a parcel prediction with phenology, vigor and a natural-language description."""
"""get_aoi_stats""","""sincrona""","""BLOCKING""","""Aggregate crop statistics (area, dominant crop, class fractions) over an AOI for a year."""
"""get_parcel_timeseries""","""sincrona""","""BLOCKING""","""Return the NDVI/NDWI/EVI time series of a parcel over a date window."""


**10 herramientas** | sincronas: **5** | diferidas: **5**

## 2. El perceiver: los modelos del equipo emiten TEXTO

El perceiver **mira** una parcela a traves de los modelos entrenados y produce una **observacion en texto plano**, nunca tensores ni probabilidades crudas hacia el reasoner. Reune el **posterior del campeon Voting-3** -- el voto ponderado de `tsvit-pheno` + `utae` + `xgb-alphaearth` (ganador de despliegue, F1-macro 0.9069 sobre `france-10`) -- resuelto sobre el fold-5 real de PASTIS via `canonical_parcel_id`, mas la **fenologia / vigor / descripcion** del descriptor (Wen et al., 2025) cuando la parcela tiene metricas fenologicas.

El metodo `to_prompt_block()` rinde esa observacion como el bloque de **anclaje** que se inyecta en el prompt del reasoner. *Ese texto* es lo que el LLM consume; la imagen y los logits nunca cruzan la frontera.

In [6]:
# Run the perceiver over the first N seeded parcels and tabulate the TEXT fields it
# exposes (no tensors). observe_parcels + perceiver_table live in ml/agent/demo.
from ml.agent.perceiver import PerceiverLayer

async with pool.acquire() as conn:
    # Session-scoped (multi-tenant): only this session's parcels.
    _rows = await conn.fetch(
        'SELECT id FROM parcels WHERE session_id = $1 ORDER BY id LIMIT $2',
        session_id, int(n_perceiver_parcels),
    )
parcel_ids = [int(r['id']) for r in _rows]

perceiver = PerceiverLayer(ctx)
observations = await demo.observe_parcels(perceiver, parcel_ids)
demo.perceiver_table(observations)

parcela,cultivo,confianza,vigor,latencia_ms
i64,str,f64,str,f64
52,"""Soft winter wheat""",0.979,"""unknown""",34666.7
53,"""Beet""",0.99,"""unknown""",8.7
54,"""Meadow""",0.951,"""unknown""",6.9


[{'parcela': 52,
  'cultivo': 'Soft winter wheat',
  'confianza': 0.979,
  'vigor': 'unknown',
  'latencia_ms': 34666.7},
 {'parcela': 53,
  'cultivo': 'Beet',
  'confianza': 0.99,
  'vigor': 'unknown',
  'latencia_ms': 8.7},
 {'parcela': 54,
  'cultivo': 'Meadow',
  'confianza': 0.951,
  'vigor': 'unknown',
  'latencia_ms': 6.9}]

In [7]:
# The actual grounding block the reasoner reads for the first parcel: plain TEXT,
# no logits -- the perceiver/reasoner contract.
first_obs = observations[0][0]
display(Markdown(
    f'**Bloque de anclaje (`to_prompt_block`) -- parcela {first_obs.parcel_id}:**'
))
print(first_obs.to_prompt_block())
display(Markdown('\n**Descripcion en lenguaje natural:**\n\n> ' + first_obs.description))

**Bloque de anclaje (`to_prompt_block`) -- parcela 52:**

Observacion del perceiver (TEXTO, sin logits):
- Cultivo estimado: Soft winter wheat (confianza 97.9%).
- Sin metricas fenologicas registradas para esta parcela.
- Vigor del cultivo: unknown.
- Clases mas probables: Soft winter wheat (97.9%), Winter rapeseed (1.5%), Beet (0.3%).
- Descripcion: Sin metricas fenologicas registradas para esta parcela.



**Descripcion en lenguaje natural:**

> Sin metricas fenologicas registradas para esta parcela.

**Lectura**: cada bloque resume lo que el modelo *ve* como frases legibles -- cultivo estimado y confianza, fenologia (inicio de verdor, pico, senescencia), vigor y las clases mas probables. El reasoner toma este texto como contexto y nunca toca el embedding ni la imagen. Asi se cumple el contrato Be My Eyes: el perceiver es los ojos, el LLM es el razonamiento.

## 3. El agente conversacional, de principio a fin

Construimos el agente con el reasoner por defecto y le hacemos **preguntas reales**. El agente decide que herramientas llamar, las ejecuta sobre la base de la sesion y redacta la respuesta. El ayudante `demo.run_agent_turn` recorre `stream_response` y renderiza el flujo `tool_call -> tool_result -> respuesta`, de modo que cada cifra de la respuesta tiene un origen visible.

Estas tres consultas ejercitan las herramientas **sincronas** que dependen de una parcela: `list_parcels`, `explain_prediction` y `get_parcel_timeseries`.

In [8]:
# Build the agent on the default reasoner and put three real questions to it. The
# driver (ml/agent/demo) renders the tool_call -> tool_result -> answer flow.
from ml.agent.agent import create_agent

agent = create_agent(model=model, settings=settings)
print('agente listo | backend:', type(agent.backend).__name__,
      '| modelo:', getattr(agent.backend, 'model', None),
      '| herramientas:', [t.name for t in agent.tools])

_pid = parcel_ids[0]
_live_queries = [
    ('inventario -> list_parcels',
     'Cuantas parcelas tengo y de que cultivos son? Dame un resumen.'),
    ('explicacion -> explain_prediction',
     f'Explica la prediccion de la parcela {_pid}: que cultivo es, con que confianza '
     'y que dice su fenologia.'),
    ('serie temporal -> get_parcel_timeseries',
     f'Como evoluciono el NDVI de la parcela {_pid} durante 2019? Resume su '
     'comportamiento estacional.'),
]
for _label, _q in _live_queries:
    await demo.run_agent_turn(
        agent, _q, ctx=ctx, session_id=session_id, title=f'### {_label}'
    )

agente listo | backend: GeminiBackend | modelo: gemini-3.5-flash | herramientas: ['list_parcels', 'get_parcel_timeseries', 'get_aoi_stats', 'classify_new_parcel', 'explain_prediction']


### inventario -> list_parcels

> Cuantas parcelas tengo y de que cultivos son? Dame un resumen.

**herramienta** `list_parcels` | argumentos: `{"session_id": "f1417dbc-3053-413e-8c5a-b66f9b748c0d"}`

**resultado** (ok) `list_parcels`: `{"parcels": [{"parcel_id": 52, "crop_class": "Soft winter wheat", "confidence": 0.97939532995224}, {"parcel_id": 53, "crop_class": "Beet", "confidence": 0.9903580546379089}, {"parcel_id": 54, "crop_class": "Meadow", "confidence": 0.9513372778892517}, {"parcel_id": 55, "crop_class": "Winter rapeseed", "confidence": 0.79 ...`

#### Respuesta del reasoner

Actualmente dispone de un total de **9 parcelas** en su sesión. A continuación, le presento el resumen detallado de los cultivos identificados para cada una de ellas, obtenido a través de la herramienta de listado de parcelas (`list_parcels`):

| ID de Parcela | Cultivo Identificado (Clase) | Confianza de la Predicción |
| :---: | :--- | :---: |
| **52** | Trigo blando de invierno (*Soft winter wheat*) | **97.94%** |
| **53** | Remolacha (*Beet*) | **99.04%** |
| **54** | Pradera (*Meadow*) | **95.13%** |
| **55** | Colza de invierno (*Winter rapeseed*) | **79.71%** |
| **56** | Maíz (*Corn*) | **29.05%** |
| **57** | Cebada de invierno (*Winter barley*) | **67.85%** |
| **58** | Girasol (*Sunflower*) | **92.30%** |
| **59** | Viñedo (*Grapevine*) | **54.72%** |
| **60** | Soja (*Soybeans*) | **37.69%** |

### Análisis del resumen:
* **Predicciones de Alta Confianza (>90%):** Las parcelas **53 (Remolacha)**, **52 (Trigo blando de invierno)**, **54 (Pradera)** y **58 (Girasol)** muestran clasificaciones sumamente robustas.
* **Predicciones Moderadas (50% - 80%):** La colza de invierno (parcela 55), la cebada de invierno (parcela 57) y el viñedo (parcela 59) muestran niveles de confianza aceptables, pero podrían beneficiarse de un seguimiento fenológico.
* **Predicciones de Baja Confianza (<50%):** El maíz (parcela 56) y la soja (parcela 60) presentan niveles de confianza muy bajos. Para estas parcelas, se recomienda solicitar un análisis temporal detallado o una explicación de la predicción para evaluar las curvas de índices de vegetación (como NDVI) y descartar posibles confusiones espectrales entre cultivos de verano. 

¿Desea que analicemos la serie temporal o la explicación de la predicción de alguna parcela en específico?

_herramientas: ['list_parcels'] | latencia: 8908.2 ms_

...

[warning  ] agent_tool_args_invalid        errors=2 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d tool=get_parcel_timeseries


**error del agente**: argumentos invalidos para get_parcel_timeseries: 2 error(es)

[warning  ] agent_loop_max_turns           max_turns=8 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


**error del agente**: el agente no produjo una respuesta final tras 8 turnos de herramientas

#### Respuesta del reasoner

_(sin texto)_

_herramientas: ['explain_prediction', 'list_parcels'] | latencia: 63327.5 ms_

### Las herramientas de area y diferidas, en crudo

El bucle conversacional de Gemini no puede invocar las herramientas **diferidas** (su `behavior=NON_BLOCKING` lo rechaza la API estandar de generacion). Las demostramos **directamente** -- el mismo `tool_result` que consumiria el agente -- sobre un AOI pequeno alrededor de la primera parcela: `classify_new_parcel`, `get_aoi_stats`, `compare_models` y `add_aoi`. El ayudante `demo.run_tool` inyecta el `session_id` y **degrada de forma honesta**.

`compare_models` contrasta los **tres miembros del Voting-3 + FarSLIP** (cada uno con su OOF fold-5 real), resuelto por el `canonical_parcel_id` de la parcela: cuando los miembros coinciden el acuerdo es 1.0, y cuando discrepan se ve el valor real del ensamble. `classify_new_parcel` sobre un AOI nuevo devuelve `needs_gee_sampling` (la parcela aun no tiene embedding AlphaEarth materializado: el agente dispararia el muestreo GEE) -- es la ruta honesta para un poligono recien dibujado.

In [9]:
# Demonstrate the AOI-based + deferred tools DIRECTLY (run_tool injects the session id
# and degrades honestly). A tiny AOI around the first parcel centroid.
async with pool.acquire() as conn:
    _c = await conn.fetchrow(
        'SELECT ST_X(ST_Centroid(geom)) AS lon, ST_Y(ST_Centroid(geom)) AS lat '
        'FROM parcels WHERE id = $1', _pid
    )
_lon, _lat = float(_c['lon']), float(_c['lat'])
_d = 0.002
_aoi = {'type': 'Polygon', 'coordinates': [[
    [_lon - _d, _lat - _d], [_lon + _d, _lat - _d], [_lon + _d, _lat + _d],
    [_lon - _d, _lat + _d], [_lon - _d, _lat - _d]]]}

await demo.run_tool('classify_new_parcel',
                    {'aoi': _aoi, 'year': classify_year, 'model': classify_model}, ctx)
await demo.run_tool('get_aoi_stats', {'aoi': _aoi, 'year': classify_year}, ctx)
# compare_models contrasts the three Voting-3 members + FarSLIP (its own real
# fold-5 OOF) for the parcel, resolved by its stored canonical PASTIS-R id.
await demo.run_tool('compare_models',
                    {'parcel_id': _pid,
                     'models': ['tsvit-pheno', 'utae', 'xgb-alphaearth',
                                'farslip-ft18']}, ctx)
await demo.run_tool('add_aoi', {'aoi': _aoi, 'name': f'Demo AOI parcela {_pid}'}, ctx)

**`classify_new_parcel`** -> `{"crop_class": "needs_gee_sampling", "confidence": 0.05555555555555555, "class_probabilities": {"needs_gee_sampling": 1.0}}`

**`get_aoi_stats`** -> `{"area_ha": 12.94180380923748, "dominant_crop": "Winter rapeseed", "crop_fractions": {"Beet": 0.2, "Corn": 0.2, "Meadow": 0.2, "Soft winter wheat": 0.2, "Winter rapeseed": 0.2}, "n_parcels": 5}`

**`compare_models`** -> `{"parcel_id": 52, "predictions": {"tsvit-pheno": "Soft winter wheat", "utae": "Soft winter wheat", "xgb-alphaearth": "Soft winter wheat", "farslip-ft18": "Soft winter wheat"}, "agreement": 1.0}`

**`add_aoi`** -> `{"aoi_id": 9, "label": "Demo AOI parcela 52", "area_ha": 12.941803932189941}`

{'name': 'add_aoi',
 'ok': True,
 'result': {'aoi_id': 9,
  'label': 'Demo AOI parcela 52',
  'area_ha': 12.941803932189941},
 'error': None}

**Lectura**: en cada turno el agente **primero actua** (una o mas llamadas a herramientas sobre la base real) y **luego responde**. Las herramientas de area y diferidas devuelven el mismo texto estructurado que el reasoner consume. Las dos restantes (`search_stac` y `get_tiles`) dependen de servicios externos (catalogo STAC / CDSE y TiTiler) y corren via el *worker*; no se invocan aqui para no exigir esos servicios en la demo. Toda cifra tiene origen en un `tool_result` visible: no hay numeros inventados.

## 4. Spatial-RAG *lite*: anclaje en parcelas vecinas reales

Para reducir alucinaciones, el agente puede anclarse en un corpus de documentos reales (descripciones fenologicas de parcelas PASTIS-R) cercanos al area consultada. La capa *lite* combina **en serie** un prefiltro **espacial** (`ST_DWithin` sobre geografia) y una busqueda **semantica** (coseno con pgvector sobre el embedding AlphaEarth de 64 dimensiones), fusionados con un peso configurable. Primero recuperamos; luego el **reasoner razona sobre esos vecinos** (el RAG en uso).

In [10]:
# Real RAG corpus glimpse + a real retrieval near the first parcel. spatial_rag runs
# the lite pipeline; rag_table (ml/agent/demo) renders the fused score + distance.
from ml.agent.rag import spatial_rag

async with pool.acquire() as conn:
    _rag_total = await conn.fetchval('SELECT count(*) FROM rag_documents')
    _rag_geom = await conn.fetchval(
        'SELECT count(*) FROM rag_documents WHERE geom IS NOT NULL'
    )
display(Markdown(
    f'Corpus RAG: **{_rag_total}** documentos ({_rag_geom} con geometria para el '
    'prefiltro espacial).'
))

# The session's parcels are real PASTIS-R (same region as the corpus), so the AOI
# around the first parcel centroid has real neighbours within the radius.
aoi_rag = {'type': 'Polygon', 'coordinates': [[
    [_lon - 0.05, _lat - 0.05], [_lon + 0.05, _lat - 0.05], [_lon + 0.05, _lat + 0.05],
    [_lon - 0.05, _lat + 0.05], [_lon - 0.05, _lat - 0.05]]]}
retrieved = await spatial_rag(
    ctx,
    query='Que cultivos y fenologia hay en las parcelas vecinas a esta area?',
    aoi=aoi_rag, top_k=int(rag_top_k), radius_m=float(rag_radius_m),
)
demo.rag_table(retrieved)

Corpus RAG: **300** documentos (300 con geometria para el prefiltro espacial).

doc_id,fuente,parcela,distancia_m,score,contenido
i64,str,str,f64,f64,str
65,"""phenology_caption""","""10021_7""",0.0,1.0,"""La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u..."""
24,"""phenology_caption""","""10009_5""",0.0,0.9926,"""La temporada agronomica probable se inicia tempranamente, mostrando un crecimiento inicial..."""
61,"""phenology_caption""","""10021_13""",0.0,0.9886,"""La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u..."""
66,"""phenology_caption""","""10021_8""",0.0,0.9884,"""La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u..."""
25,"""phenology_caption""","""10009_50""",0.0,0.9883,"""La curva NDVI sugiere una temporada agronomica de invierno, con un inicio de crecimiento t..."""


[{'doc_id': 65,
  'fuente': 'phenology_caption',
  'parcela': '10021_7',
  'distancia_m': 0.0,
  'score': 1.0,
  'contenido': 'La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u...'},
 {'doc_id': 24,
  'fuente': 'phenology_caption',
  'parcela': '10009_5',
  'distancia_m': 0.0,
  'score': 0.9926,
  'contenido': 'La temporada agronomica probable se inicia tempranamente, mostrando un crecimiento inicial...'},
 {'doc_id': 61,
  'fuente': 'phenology_caption',
  'parcela': '10021_13',
  'distancia_m': 0.0,
  'score': 0.9886,
  'contenido': 'La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u...'},
 {'doc_id': 66,
  'fuente': 'phenology_caption',
  'parcela': '10021_8',
  'distancia_m': 0.0,
  'score': 0.9884,
  'contenido': 'La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y u...'},
 {'doc_id': 25,
  'fuente': 'phenology_caption',
  'parcela': '10009_50',
  'distancia_m': 0.0

In [11]:
# Spatial-RAG lite IN USE: feed the retrieved neighbours to the reasoner as a cited
# grounding block and let it answer over THAT text. Honest: skipped cleanly if the
# default reasoner is not reachable here (no fabricated answer).
_avail_default, _ = demo.probe_availability([model], settings, display=False)
if retrieved and _avail_default.get(model):
    _ctx_block = '\n'.join(f'[{d.source}:{d.parcel_id}] {d.content}' for d in retrieved)
    _rag_q = (
        'Contexto recuperado por Spatial-RAG (parcelas vecinas reales):\n'
        f'{_ctx_block}\n\nCon SOLO ese contexto, resume en dos frases que cultivos y '
        'fenologia predominan en las parcelas vecinas y cita [fuente:parcela].'
    )
    await demo.run_backend_turn(
        model, _rag_q, settings=settings, ctx=ctx, session_id=session_id,
        availability=_avail_default,
        title='### El reasoner razona sobre el contexto del RAG',
    )
elif not retrieved:
    display(Markdown('> Sin vecinos en el radio: nada que anclar. Aumenta `rag_radius_m`.'))
else:
    display(Markdown(
        f'> Reasoner `{model}` no disponible aqui; la celda se ejecuta con credenciales '
        '/ endpoint vivos. El bloque de contexto de arriba es lo que recibiria el '
        'reasoner.'
    ))

### El reasoner razona sobre el contexto del RAG

> Contexto recuperado por Spatial-RAG (parcelas vecinas reales):
[phenology_caption:10021_7] La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y un pico de actividad vegetal alto alcanzado a mediados de año. El crecimiento parece ser relativamente rápido, seguido de una senescencia marcada, lo que indica una madurez de duración media.
[phenology_caption:10009_5] La temporada agronomica probable se inicia tempranamente, mostrando un crecimiento inicial rápido hasta alcanzar un pico de actividad alto. Posteriormente, se observa una senescencia marcada, seguida de un resurgimiento tardío que sugiere una madurez de duración media.
[phenology_caption:10021_13] La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y una senescencia marcada. El crecimiento parece ser relativamente rápido, alcanzando un pico alto de actividad vegetativa. La duración de la madurez se estima como media, dada la ventana de tiempo entre el inicio del crecimiento y la senescencia.
[phenology_caption:10021_8] La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y un pico de desarrollo alto y robusto. El crecimiento parece ser relativamente uniforme hasta alcanzar su máximo, seguido de una senescencia marcada. La duración estimada de la madurez se percibe como media, dada la ventana de tiempo entre el inicio del crecimiento y la senescencia.
[phenology_caption:10009_50] La curva NDVI sugiere una temporada agronomica de invierno, con un inicio de crecimiento temprano y un pico de alta actividad vegetativa alcanzado a mediados de año. El comportamiento del crecimiento parece ser uniforme, con un único pico pronunciado, indicando una madurez de duración media. La senescencia se inicia a finales del verano, culminando en un estado de bajo vigor vegetativo hacia el final del ciclo.

Con SOLO ese contexto, resume en dos frases que cultivos y fenologia predominan en las parcelas vecinas y cita [fuente:parcela].

#### Respuesta del reasoner

En las parcelas vecinas predomina un ciclo de cultivo de invierno caracterizado por un inicio temprano y un crecimiento rápido y uniforme hacia un pico de alta actividad vegetal a mitad de año [fuente:10021_7, 10009_5, 10021_13, 10021_8, 10009_50]. Posteriormente, se observa una senescencia marcada que define una madurez de duración media [fuente:10021_7, 10021_13, 10021_8, 10009_50], registrándose en casos particulares un resurgimiento tardío [fuente:10009_5] o la finalización del ciclo con bajo vigor hacia el final del verano [fuente:10009_50].

_herramientas: ninguna | latencia: 8730.0 ms_

**Lectura**: el reasoner recibe los documentos como bloque de contexto citado (`[fuente:parcela] ...`). Son **parcelas vecinas reales**, no ejemplos genericos: el modelo se aterriza en evidencia local y cita de donde sale cada afirmacion. Esa es la palanca anti-alucinacion del patron Spatial-RAG.

## 5. Tres reasoners, un mismo perceiver: comparacion de backends

La abstraccion `LLMBackend` / `make_backend` desacopla el bucle del agente del modelo concreto. Contrastamos **tres reasoners**:

- **Gemini 3.5 Flash** -- nube (`GeminiBackend`, Vertex AI o la API GenAI).
- **Qwen3.6-VL** -- on-prem **multimodal** servido por llama.cpp (`:8003`).
- **Qwen3.5-35B** -- on-prem **texto** servido por vLLM (`:8002`). Soberania de datos: el razonamiento ocurre dentro del perimetro.

Primero resolvemos cada nombre a su backend (sin red) y **sondeamos honestamente** cual esta vivo aqui (credenciales de Gemini, o endpoint OpenAI-compatible que responda). El perceiver es **el mismo** para los tres: la clase y la confianza no cambian con el LLM, porque el LLM no clasifica.

> Los dos Qwen on-prem comparten la **misma H100**, asi que se evaluan **uno a la vez** (no caben los dos en VRAM a la vez). Cada corrida en vivo guarda su registro real bajo `reports/copilot_backends/` y la tabla final se rearma con todas las corridas: un backend que aun no se evaluo aparece como **no disponible**, sin inventar.

In [12]:
# Resolve each reasoner to its backend (network-free) and probe availability HONESTLY.
# Both renderings live in ml/agent/demo.
demo.backend_overview(backend_models, settings)
availability, _ = demo.probe_availability(backend_models, settings)

modelo,etiqueta,backend,modelo_servido,endpoint
str,str,str,str,str
"""gemini-3.5-flash""","""Gemini 3.5 Flash (nube, Vertex AI / GenAI)""","""GeminiBackend""","""gemini-3.5-flash""","""Vertex AI / GenAI"""
"""qwen3.6-vl""","""Qwen3.6-VL (on-prem multimodal, llama.cpp :8003)""","""OllamaBackend""","""qwen36-vl""","""http://127.0.0.1:8003/v1"""
"""qwen35""","""Qwen3.5-35B (on-prem texto, vLLM :8002)""","""VLLMOpenAIBackend""","""qwen35""","""http://127.0.0.1:8002/v1"""


modelo,disponible,motivo
str,str,str
"""gemini-3.5-flash""","""si""","""GEMINI_API_KEY presente en .env.local"""
"""qwen3.6-vl""","""si""","""HTTP 200 en http://127.0.0.1:8003/v1/models"""
"""qwen35""","""NO""","""sin respuesta en http://127.0.0.1:8002/v1/models: timed out"""


**Backends disponibles:** `gemini-3.5-flash`, `qwen3.6-vl`

In [13]:
# Put the SAME grounded question (the perceiver's TEXT for the first parcel) to every
# available backend. The dense observation is FIXED; only the reasoning over it varies.
# The on-prem Qwen text and Qwen-VL share the single H100 GPU, so they are evaluated
# ONE AT A TIME: each live pass saves its real record under reports/copilot_backends/
# and the table is reassembled from every pass (a backend never run shows as such).
_grounded_q = (
    first_obs.to_prompt_block()
    + '\n\nCon esa observacion del perceiver (no inventes cifras), di en una frase '
    f'breve que cultivo es la parcela {first_obs.parcel_id}, su confianza y su vigor.'
)
_records_dir = REPO_ROOT / 'reports' / 'copilot_backends'
_this_run = {}
for _name in backend_models:
    _rec = await demo.run_backend_turn(
        _name, _grounded_q, settings=settings, ctx=ctx, session_id=session_id,
        availability=availability,
    )
    _this_run[_name] = _rec
    demo.save_backend_record(_rec, _records_dir)   # persists only a successful run
# Prefer a persisted real record (possibly from a previous one-at-a-time pass).
_persisted = demo.load_persisted_records(backend_models, _records_dir)
backend_records = [_persisted.get(_n, _this_run[_n]) for _n in backend_models]
demo.cross_backend_table(backend_records)

### Backend `gemini-3.5-flash` -- Gemini 3.5 Flash (nube, Vertex AI / GenAI) (GeminiBackend)

> Observacion del perceiver (TEXTO, sin logits):
- Cultivo estimado: Soft winter wheat (confianza 97.9%).
- Sin metricas fenologicas registradas para esta parcela.
- Vigor del cultivo: unknown.
- Clases mas probables: Soft winter wheat (97.9%), Winter rapeseed (1.5%), Beet (0.3%).
- Descripcion: Sin metricas fenologicas registradas para esta parcela.

Con esa observacion del perceiver (no inventes cifras), di en una frase breve que cultivo es la parcela 52, su confianza y su vigor.

#### Respuesta del reasoner

La parcela 52 está clasificada como trigo blando de invierno (*soft winter wheat*) con una confianza del 97.9% y un nivel de vigor desconocido (*unknown*).

_herramientas: ninguna | latencia: 2994.8 ms_

### Backend `qwen3.6-vl` -- Qwen3.6-VL (on-prem multimodal, llama.cpp :8003) (OllamaBackend)

> Observacion del perceiver (TEXTO, sin logits):
- Cultivo estimado: Soft winter wheat (confianza 97.9%).
- Sin metricas fenologicas registradas para esta parcela.
- Vigor del cultivo: unknown.
- Clases mas probables: Soft winter wheat (97.9%), Winter rapeseed (1.5%), Beet (0.3%).
- Descripcion: Sin metricas fenologicas registradas para esta parcela.

Con esa observacion del perceiver (no inventes cifras), di en una frase breve que cultivo es la parcela 52, su confianza y su vigor.

#### Respuesta del reasoner

La parcela 52 se clasifica como Soft winter wheat con una confianza del 97.9%, y su vigor se reporta como unknown.

_herramientas: ninguna | latencia: 1951.9 ms_

> Backend `qwen35` **no disponible**; turno omitido.

backend,disponible,respondio,n_herramientas,herramientas,latencia_ms,chars_respuesta
str,str,str,i64,str,f64,i64
"""gemini-3.5-flash""","""si""","""si""",0,"""-""",2994.8,155
"""qwen36-vl""","""si""","""si""",0,"""-""",1951.9,114
"""qwen35""","""si""","""si""",0,"""-""",2391.1,174


Backends que completaron el turno: `gemini-3.5-flash`, `qwen36-vl`, `qwen35`. El perceiver entrega la misma clase/confianza a todos; el LLM solo razona sobre ese TEXTO (Be My Eyes).

[{'backend': 'gemini-3.5-flash',
  'disponible': 'si',
  'respondio': 'si',
  'n_herramientas': 0,
  'herramientas': '-',
  'latencia_ms': 2994.8,
  'chars_respuesta': 155},
 {'backend': 'qwen36-vl',
  'disponible': 'si',
  'respondio': 'si',
  'n_herramientas': 0,
  'herramientas': '-',
  'latencia_ms': 1951.9,
  'chars_respuesta': 114},
 {'backend': 'qwen35',
  'disponible': 'si',
  'respondio': 'si',
  'n_herramientas': 0,
  'herramientas': '-',
  'latencia_ms': 2391.1,
  'chars_respuesta': 174}]

**Lectura**: con el perceiver fijo, lo que cambia entre backends es la **calidad del razonamiento sobre ese texto**, el uso de herramientas y la **latencia / coste**. Gemini y Qwen3.5 (texto) pueden ademas llamar herramientas; Qwen3.6-VL razona sobre el texto inyectado. Un backend no disponible aparece como tal -- sin cifras inventadas. Construir el agente con cualquiera de los tres produce el mismo flujo de esta demo, pero razonando dentro (on-prem) o fuera (nube) del perimetro del cliente.

## Conclusiones

**Que se demostro**

- Un **copiloto conversacional completo** que responde preguntas sobre parcelas agricolas reales, hablando con un LLM que **razona** pero no clasifica pixeles.
- La **separacion Be My Eyes**: los modelos del equipo miran cada parcela y emiten una observacion en texto; el LLM lee ese texto, llama herramientas y redacta la respuesta.
- Un **conjunto cerrado de diez herramientas** con esquemas validados: las cinco sincronas y, en crudo, las de area y diferidas (`classify`, `get_aoi_stats`, `compare_models`, `add_aoi`).
- Un **RAG espacial en uso**: el reasoner se ancla en parcelas vecinas reales, recuperadas combinando cercania geografica y similitud del embedding satelital, y cita su origen -- la palanca anti-alucinacion.
- **Tres reasoners intercambiables** (Gemini nube, Qwen3.6-VL y Qwen3.5 on-prem) sobre el **mismo** perceiver, con sonda honesta de disponibilidad y la misma pregunta anclada para cada uno.

**Lo que sigue**

- Activar el RAG y las herramientas diferidas dentro del bucle del agente via el ejecutor en segundo plano, para que el reasoner pida contexto vecino por su cuenta.
- Conectar el frontend de mapa para dibujar areas y disparar estas mismas consultas.
- Para el transfer learning Francia -> Italia accedido por el copiloto, ver `notebooks/transfer/us079_copilot_original_vs_tl` (vista copiloto) y `us079_transfer_italia_eval` (analisis denso).

### Cierre

Cerramos el *pool* de conexiones de forma ordenada al terminar la demostracion.

In [14]:
# Close the shared asyncpg pool cleanly at the end of the demo.
from ml.agent.db import close_pool

await close_pool()
print('pool cerrado.')

pool cerrado.
